In [1]:
from myTransformer import *
from pedalboard_param_utils import *

#dac_model_path = dac.utils.download(model_type="44khz")
#dac_model = dac.DAC.load(dac_model_path, weights_only=True).to('cuda')
model_dim = 128
key_dim = 64
num_heads = 8 
enc_out_dim = 9
ffn_hidden_dim = 512
num_stack=8
num_vocab = len(vocab)

my_model = MyModel(
    dac=None, 
    num_vocab=num_vocab, 
    model_dim=model_dim, 
    key_dim=key_dim, 
    enc_out_dim=None, 
    ffn_hidden_dim=ffn_hidden_dim, 
    num_heads=num_heads,
    num_stack=num_stack
).to("cuda")

my_model.parameters()

def count_parameters(m: torch.nn.Module, trainable: bool = False):
    if trainable:
        return sum([p.shape.numel() for p in m.parameters() if p.requires_grad])
    else:
        return sum([p.shape.numel() for p in m.parameters()])

count_parameters(my_model)

2026-02-12 16:48:30.659208: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-12 16:48:30.731170: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-12 16:48:30.746542: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-12 16:48:31.015462: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: li

2389843

In [2]:
num_vocab

82

In [3]:
from EGFxSetDataParam import *
from ICMTSMTGuitarDataParam import * 
from torch.utils.data import ConcatDataset, DataLoader

egfx_data = EGFxSetData(pedal_dict)
icmt_mono = ICMTSMTGuitarDataMono(pedal_dict)
icmt_poly = ICMTSMTGuitarDataPoly(pedal_dict)

batch_size = 32

combined = ConcatDataset([egfx_data, icmt_mono])
loader = DataLoader(dataset=combined, batch_size=32, num_workers=4, shuffle=True, collate_fn=collate_data)


In [4]:
from torch.optim import Adam
from torch.nn import CrossEntropyLoss

lr = 0.0002
optimizer = Adam(my_model.parameters(), lr=lr)
criterion = CrossEntropyLoss()


In [5]:
from torch.utils.tensorboard import SummaryWriter
import datetime 
torch.manual_seed(0)

run_dir = "logs"
!rm -rf ./logs
run_time = datetime.datetime.now().strftime("%I:%M%p on %B %d, %Y")
logger = SummaryWriter(log_dir=Path(run_dir) / run_time, flush_secs=20)
epochs = 50
device = "cuda"

In [ ]:
for epoch in range(epochs):
    
    # weight batch losses/scores proportional to batch size
    iter_count = 0
    loss_epoch = 0
    
    for batch_idx, batch_data in enumerate(loader):
        audio, pedal_str = batch_data
        my_model.zero_grad()
        
        # train on a batch of inputs
        logits = my_model(audio.to(device), pedal_str[:, :-1].to(device)).transpose(1,2)
        loss_batch = criterion(logits, pedal_str[:, 1:].to(device))
        loss_batch.backward()
        optimizer.step()
        
        # log loss
        loss_epoch += loss_batch.detach().item() * batch_size
        iter_count += batch_size
    
    # plot loss
    loss_epoch /= iter_count
    logger.add_scalar("cross_entropy_loss", loss_epoch, epoch)
    
    if not epoch % 5:
        print(f"Epoch: {epoch}\tCross Entropy Loss: {loss_epoch :0.4f}")

Epoch: 0	Cross Entropy Loss: 2.3382
Epoch: 5	Cross Entropy Loss: 0.8384
Epoch: 10	Cross Entropy Loss: 0.7661
Epoch: 15	Cross Entropy Loss: 0.7417
Epoch: 20	Cross Entropy Loss: 0.7171
Epoch: 25	Cross Entropy Loss: 0.7096


In [ ]:
loader_iter = iter(loader)
source, target = next(loader_iter)
predictions = torch.tensor([vocab.to_num(["<start>"]) for _ in range(32)]).to(device)
max_len = 32
while (predictions.shape[-1] < max_len):  
    logits = my_model(source.to(device), target.to(device))[:, -1:, :]
    next_word_indices = torch.argmax(logits, dim=-1) # [batch_size, 1, num_vocabs]
    
    predictions = torch.cat((predictions, next_word_indices), dim=1)



In [ ]:
for prediction in predictions.detach(): 
    print(vocab.to_str(prediction.cpu().numpy()))